# 补充：逻辑斯蒂回归与优化

对应本地《深度强化学习》第 1 章 1.1.2 节“逻辑斯蒂回归”（书内第 4–6 页，PDF 第 14–16 页），并补讲前一节出现的“优化问题”。这一篇只回答两件事：模型怎样把输入变成二分类预测？训练时究竟在优化什么？

先读概念和手算，不需要写训练代码。


## 1. 先分清任务：预测数值还是判断类别？

线性回归的目标可以是任意连续数值，例如房价。现在换成二分类：标签 $y\in\{0,1\}$。书中用血检指标作为输入，标签表示某个二元结果；我们把它当成理解模型的例子，而不是医学判断工具。

假设一个样本有 $d$ 个特征，写成列向量 $\boldsymbol{x}\in\mathbb{R}^d$。想要的输出是“属于类别 1 的预测概率” $p$。如果直接使用线性式 $\boldsymbol{x}^{\top}\boldsymbol{w}+b$，结果可能小于 0 或大于 1；因此还需要一个把实数压到 0 与 1 之间的函数。


## 2. 逻辑斯蒂回归模型做了哪两步？

第一步计算一个实数分数，常称为 logit：

$$
z=\boldsymbol{x}^{\top}\boldsymbol{w}+b
$$

第二步用 Sigmoid 映射到 $(0,1)$：

$$
p=\sigma(z)=\frac{1}{1+e^{-z}}
$$

这里 $\boldsymbol{w}$ 和 $b$ 是**需要训练的参数**；$\boldsymbol{x}$ 是一个样本的已知特征。输出 $p$ 是模型对类别 1 的预测概率，$1-p$ 对应类别 0。简单核对：$z=0$ 时 $p=0.5$；$z$ 越大，$p$ 越靠近 1。

| $z$ | $-2$ | $0$ | $2$ |
| ---: | ---: | ---: | ---: |
| $\sigma(z)$（约） | $0.119$ | $0.500$ | $0.881$ |

如果需要输出类别，可以在预测阶段选阈值，例如 $p\geq0.5$ 判为 1。**阈值决定怎样报告类别，不是训练时的优化变量。**


### 为什么叫“回归”，却用于分类？

名称来自模型对 logit 的线性表达式；最终输出经 Sigmoid 变成二元预测概率，所以通常用于**二分类**。虽然 Sigmoid 是非线性的，当阈值取 $0.5$ 时，分类边界对应 $z=0$，仍是输入空间中的线性边界。

它能给出介于 0 和 1 之间的数，但这些数是否与真实发生频率一致，还要看数据、训练和概率校准。这里先把 $p$ 当作模型的预测值，不把它当作已经验证的医学风险。


## 3. 什么叫优化？

**优化**是：先明确哪些量可以改变，再定义一个衡量好坏的目标，然后寻找让目标尽量小或尽量大的取值。

在逻辑斯蒂回归中：

| 问题 | 对应对象 |
| --- | --- |
| 已知且固定的是什么？ | 训练样本 $(\boldsymbol{x}_i,y_i)$ |
| 可以调整的是什么？ | 参数 $\boldsymbol{w},b$ |
| 怎样评价一组参数？ | 计算训练数据上的损失，以及可能加入的正则项 |
| 想找到什么？ | 使目标函数尽量小的参数 |

因此，训练不是“改动输入样本”，也不是“直接改动预测概率”；训练通过改变 $\boldsymbol{w},b$，间接改变模型输出。


### 先用一个不涉及分类的小例子看优化

设目标函数为 $J(w)=(w-3)^2$。我们可以改变 $w$；当 $w=3$ 时，$J(w)$ 的最小值是 0。

$$
\min_w J(w)=0,
\qquad
\underset{w}{\arg\min}\,J(w)=3
$$

**min 返回最小的目标值，argmin 返回达到最小值的变量取值。**训练真实模型时，目标远比这个抛物线复杂，通常无法直接看出最优参数，需要用迭代算法逐步寻找。


## 4. 怎样判断一次二分类预测的好坏？

模型输出 $p$，真实标签是 $y$。书中用交叉熵比较真实的二元分布 $(y,1-y)$ 与预测分布 $(p,1-p)$。对一个样本，写成更常见的形式就是**二元交叉熵**：

$$
\ell(y,p)=-y\log p-(1-y)\log(1-p)
$$

若 $y=1$，它变成 $-\log p$：预测 $p$ 越接近 1，损失越小。若 $y=0$，它变成 $-\log(1-p)$：预测 $p$ 越接近 0，损失越小。

| 真实标签 | 预测 $p$ | 单样本损失（约） | 解释 |
| ---: | ---: | ---: | --- |
| $1$ | $0.9$ | $0.105$ | 与标签一致，惩罚小 |
| $1$ | $0.1$ | $2.303$ | 自信地预测错，惩罚大 |
| $0$ | $0.1$ | $0.105$ | 与标签一致，惩罚小 |
| $0$ | $0.9$ | $2.303$ | 自信地预测错，惩罚大 |


### 为什么不直接优化“分类正确率”？

若把 $p$ 用阈值变成 0 或 1，再数答对几个，指标会在很多参数取值区间内保持不变。它不容易提供“参数往哪个方向改一点会更好”的梯度信息。

交叉熵直接使用连续的预测值 $p$，能区分“错得有多自信”。它适合用梯度法训练。正确率仍有用，但通常放在**评估**环节，与训练损失分工不同。


## 5. 从单样本损失到完整优化问题

训练集有 $n$ 个样本。先把各样本的损失取平均：

$$
L(\boldsymbol{w},b)
=\frac{1}{n}\sum_{i=1}^{n}
\ell\!\left(y_i,\sigma(\boldsymbol{x}_i^\top\boldsymbol{w}+b)\right)
$$

书中还加入正则项 $R(\boldsymbol{w})$，写成：

$$
(\boldsymbol{w}^{\star},b^{\star})
=\underset{\boldsymbol{w},b}{\arg\min}\,
\left[L(\boldsymbol{w},b)+R(\boldsymbol{w})\right]
$$

$L$ 衡量对训练样本的拟合误差；$R$ 约束参数，防止只靠把权重变得很大来拟合数据。例如常见选择是 $R(\boldsymbol{w})=\frac{\lambda}{2}\|\boldsymbol{w}\|_2^2$。**损失函数**常指 $L$；加上正则项后的整体是这里真正被最小化的**目标函数**。


## 6. 优化算法怎样寻找较好的参数？

确定目标函数后，还需要一个求解办法。梯度下降从一组初始参数出发，反复计算目标对参数的梯度，再往降低目标的方向走一小步：

$$
\boldsymbol{w}_{\mathrm{new}}
=\boldsymbol{w}_{\mathrm{old}}-\alpha\nabla_{\boldsymbol{w}}J,
\qquad
b_{\mathrm{new}}
=b_{\mathrm{old}}-\alpha\frac{\partial J}{\partial b}
$$

这里 $J=L+R$ 是目标函数，$\alpha>0$ 是学习率。梯度回答“局部往哪边变大、变化有多快”；前面的负号表示我们要往目标变小的方向更新。

**模型**定义 $p$ 怎样由输入和参数算出；**损失/目标**定义什么叫好；**优化算法**定义参数怎样逐步改变。这三件事不要混在一起。


### 梯度为什么与“预测减标签”有关？

先忽略正则项，只看一个样本。把线性分数记作 $z$，预测记作 $p=\sigma(z)$。对二元交叉熵应用链式法则，可得到：

$$
\frac{\partial \ell}{\partial z}=p-y
$$

又因为 $z=\boldsymbol{x}^{\top}\boldsymbol{w}+b$，所以：

$$
\nabla_{\boldsymbol{w}}\ell=(p-y)\boldsymbol{x},
\qquad
\frac{\partial \ell}{\partial b}=p-y
$$

直觉上，若真实标签为 1 而预测 $p$ 太小，则 $p-y<0$；梯度下降会推动相关分数 $z$ 增大。若 $y=0$ 而 $p$ 太大，方向相反。对整个数据集，先汇总样本梯度，再加上正则项的梯度。


## 7. 手算一次参数更新

做一个只有一个特征、一个样本的例子：$x=2,\ y=1$；初始 $w=0,\ b=0$，学习率 $\alpha=0.1$，暂不加正则项。

1. **预测**：$z=0$，$p=\sigma(0)=0.5$。
2. **损失**：$\ell=-\log(0.5)\approx0.693$。
3. **梯度**：$\frac{\partial\ell}{\partial w}=(0.5-1)\times2=-1$；$\frac{\partial\ell}{\partial b}=0.5-1=-0.5$。
4. **更新**：$w_{\mathrm{new}}=0-0.1(-1)=0.1$，$b_{\mathrm{new}}=0-0.1(-0.5)=0.05$。
5. **再预测**：$z_{\mathrm{new}}=0.1\times2+0.05=0.25$，$p_{\mathrm{new}}\approx0.562$，新损失约为 $0.576$。

这一步使损失下降。它只是一个演示；真实训练要在许多样本上反复更新，并在独立数据上检查效果。


## 8. 训练流程与常见混淆

一轮迭代可以口头描述为：**取样本 → 算 $z$ 和 $p$ → 算损失与正则项 → 求梯度 → 更新参数**。重复后，得到一组参数，再用未参与训练的数据评估。

- Sigmoid 负责把分数映射成预测值；它本身不是优化算法。
- 交叉熵告诉模型怎样算错；它不负责改变参数。
- 梯度提供局部更新方向；学习率控制每一步的幅度。
- 预测阈值把概率变成类别；它通常不参与前述梯度更新。
- 训练损失下降不保证未见样本的效果一定提升，还需要验证。


## 自检：先口答，再回看公式

1. 一个样本的特征、真实标签、模型参数、预测概率，分别由什么符号表示？训练时改哪两个量？
2. 为什么线性分数后要接 Sigmoid？
3. 若 $y=0$ 而 $p=0.9$，二元交叉熵应大还是小？为什么？
4. 目标函数 $J=L+R$ 中，$L$ 与 $R$ 各起什么作用？
5. min 与 argmin 的结果有什么区别？
6. 重新手算上一节的更新：若把学习率从 $0.1$ 改为 $0.2$，新参数是多少？

能用自己的话把“模型 → 损失 → 优化”讲通，再进入更复杂的神经网络训练会稳得多。


## 与原书的对应

主要对应本地 drl_v1.pdf 第 1 章 1.1.2 节“逻辑斯蒂回归”（书内第 4–6 页）：二分类任务、线性函数加 Sigmoid、交叉熵，以及最小化损失加正则项。上面的数值示例、解释顺序与自检题是为当前学习阶段另行编写的。原书 PDF 留在本机，已被 Git 忽略。
